### Objects and Classes — Advanced Problems with Solutions

In this notebook we are going to keep working with the same ideas:

- classes are objects
- class objects normally have type `type`
- calling a class creates an instance
- instances know the class they were created from
- `type(...)`, `__class__`, `__name__`, and `isinstance(...)` let us inspect those relationships

But this time we are going to turn those ideas into a sequence of more advanced problems.

We will take each problem slowly, break it into smaller questions, and verify each part with code.

We are deliberately not going to rely on inheritance here.

Inheritance changes some of the details of `isinstance`, and that deserves its own discussion.

For now, we will concentrate on class objects, instances, identity, callability, introspection, and dynamic class creation.

### Problem 1 — Following an Object Through Its Entire Life Cycle

Let's start with a simple class.

Our goal is not just to create an instance.

We want to follow the relationship from the class statement all the way to the final instance object.

In [1]:
class Sensor:
    pass

At this point `Sensor` exists.

The first question is:

**What kind of object is `Sensor` itself?**

Try to predict the result before running the next cell.

In [2]:
type(Sensor)

type

So `Sensor` is a class object, and the type of that class object is `type`.

Now let's create an instance.

In [3]:
s = Sensor()

The variable `s` now refers to the object returned by calling the class.

Let's see which class Python says was used to create it.

In [4]:
type(s)

__main__.Sensor

That gives us the class object `Sensor`.

The instance also exposes its class through `__class__`.

In [5]:
s.__class__

__main__.Sensor

Those two expressions should not merely print similar-looking values.

They should actually refer to the exact same class object.

In [6]:
type(s) is s.__class__

True

Now we can state the full chain:

```text
Sensor  --> class object
type(Sensor)  --> type

s = Sensor()  --> instance
type(s)  --> Sensor
s.__class__  --> Sensor
```

Let's package the important tests together.

In [7]:
print(type(Sensor) is type)
print(type(s) is Sensor)
print(s.__class__ is Sensor)
print(type(s) is s.__class__)
print(isinstance(s, Sensor))

True
True
True
True
True


#### Solution

All five expressions evaluate to `True`.

The important part is that there are two different levels:

1. `Sensor` is a class object.
2. `s` is an instance object created by that class.

### Problem 2 — Two Instances, One Class

Now let's make the problem slightly more subtle.

Suppose we create two objects by calling the same class twice.

In [8]:
class Packet:
    pass

p1 = Packet()
p2 = Packet()

Are `p1` and `p2` the same object?

There is a temptation to say yes because they have the same type.

But object identity and object type are different questions.

In [9]:
p1 is p2

False

They are different objects.

Now let's ask whether they have the same type.

In [10]:
type(p1) is type(p2)

True

That is true.

Both type lookups return the same class object, `Packet`.

In [11]:
type(p1) is Packet

True

In [12]:
type(p2) is Packet

True

We can also compare their `__class__` attributes.

In [13]:
p1.__class__ is p2.__class__

True

So this is a useful pattern:

```text
different instances
        |
        +---- same class
```

Let's verify that each object is independently recognized as a `Packet` instance.

In [14]:
print(isinstance(p1, Packet))
print(isinstance(p2, Packet))

True
True


#### Solution

`p1 is p2` is `False`, because each call to `Packet()` created a different object.

But:

```python
type(p1) is type(p2)
```

is `True`, because both objects were created by the same class.

### Problem 3 — Class Object or Instance Object?

When looking at a value, it is very useful to ask:

> Is this value itself a class object, or is it an instance of some class?

Let's build several values and compare them.

In [15]:
class Report:
    pass

r = Report()

values = [
    Report,
    r,
    int,
    10,
    str,
    "hello",
    list,
    [],
]

A class object is normally an instance of `type`.

So we can test every value using:

```python
isinstance(value, type)
```

In [16]:
for value in values:
    print(value, "->", isinstance(value, type))

<class '__main__.Report'> -> True
<__main__.Report object at 0x000001FE190AEA50> -> False
<class 'int'> -> True
10 -> False
<class 'str'> -> True
hello -> False
<class 'list'> -> True
[] -> False


Notice the pattern.

`Report`, `int`, `str`, and `list` are class objects.

`r`, `10`, `"hello"`, and `[]` are instance objects.

Let's make that output easier to read by adding the runtime type of each value.

In [17]:
for value in values:
    print(
        repr(value),
        "| type:",
        type(value),
        "| class object:",
        isinstance(value, type)
    )

<class '__main__.Report'> | type: <class 'type'> | class object: True
<__main__.Report object at 0x000001FE190AEA50> | type: <class '__main__.Report'> | class object: False
<class 'int'> | type: <class 'type'> | class object: True
10 | type: <class 'int'> | class object: False
<class 'str'> | type: <class 'type'> | class object: True
'hello' | type: <class 'str'> | class object: False
<class 'list'> | type: <class 'type'> | class object: True
[] | type: <class 'list'> | class object: False


#### Solution

A reliable test for the specific question *"is this value a class object?"* is:

```python
isinstance(value, type)
```

For the scope of this notebook, that is the test we will use.

### Problem 4 — Why Comparing Class Names Can Be Dangerous

Suppose someone writes a type check like this:

```python
type(obj).__name__ == "Document"
```

At first this looks reasonable.

After all, class objects have a `__name__`.

But the class name is only a string.

A string does not uniquely identify a class object.

We can prove this by creating two different class objects that have the same name.

To do that we will use the three-argument form of `type`.

The form is:

```python
type(name, bases, namespace)
```

and it creates a new class object.

In [18]:
FirstDocument = type("Document", (), {})
SecondDocument = type("Document", (), {})

Let's check the names first.

In [19]:
FirstDocument.__name__

'Document'

In [20]:
SecondDocument.__name__

'Document'

The names are identical.

Now let's ask whether the class objects themselves are identical.

In [21]:
FirstDocument is SecondDocument

False

They are not.

So two distinct class objects can share the same name.

Let's create an instance of the first class.

In [22]:
doc = FirstDocument()

The name-based check would say that this object's class is called `Document`.

In [23]:
type(doc).__name__ == "Document" 

True

But now compare real class membership.

In [24]:
isinstance(doc, FirstDocument)

True

In [25]:
isinstance(doc, SecondDocument)

False

#### Solution

The first `isinstance` result is `True`.

The second is `False`.

This is why class-name strings are useful for display, but actual class objects should be used for real type checks.

### Problem 5 — Build a Better Type-Checking Function

Let's turn the previous lesson into a reusable function.

We want a function that receives:

- an object
- a class object

and checks whether the object is an instance of that class.

A first version can be very small.

In [26]:
def matches_class(obj, cls):
    return isinstance(obj, cls)

Let's test it.

In [27]:
class Invoice:
    pass

invoice = Invoice()

print(matches_class(invoice, Invoice))
print(matches_class("hello", Invoice))

True
False


But there is another problem.

What if the caller accidentally passes an instance as the second argument instead of a class?

For example, this would be a mistake:

```python
matches_class(invoice, Invoice())
```

Our helper should make that mistake easier to understand.

So before calling `isinstance`, we can verify that `cls` is itself a class object.

In [28]:
def matches_class(obj, cls):
    if not isinstance(cls, type):
        raise TypeError("cls must be a class object")

    return isinstance(obj, cls)

Now let's test both the correct case and the incorrect case.

In [29]:
print(matches_class(invoice, Invoice))

try:
    matches_class(invoice, Invoice())
except TypeError as ex:
    print(type(ex).__name__ + ":", ex)

True
TypeError: cls must be a class object


#### Solution

The final function performs two logically separate checks:

1. Is the second argument a class object?
2. Is the first argument an instance of that class?

That makes the API safer and gives a clearer error message.

### Problem 6 — Classes Are Callable, But Not Everything Callable Is a Class

We already know that calling a class creates an instance.

That means class objects are callable.

Let's verify that first.

In [30]:
class Job:
    pass

callable(Job)

True

Now compare that with an instance of the class.

In [31]:
job = Job()

callable(job)

False

The instance is not callable, because we did not define any special behavior that would make it callable.

But this gives us another question:

**Does `callable(value)` tell us whether `value` is a class?**

Let's test a normal function.

In [32]:
def add(a, b):
    return a + b

callable(add)

True

The function is callable too.

But it is not a class object.

In [33]:
isinstance(add, type)

False

So these two tests answer different questions:

```python
callable(value)
```

asks whether the value can be called.

```python
isinstance(value, type)
```

asks whether the value is a class object.

Let's compare several values side by side.

In [34]:
samples = [Job, job, add, len, int, 42]

for value in samples:
    print(
        repr(value),
        "| callable:",
        callable(value),
        "| class object:",
        isinstance(value, type)
    )

<class '__main__.Job'> | callable: True | class object: True
<__main__.Job object at 0x000001FE190AEF90> | callable: False | class object: False
<function add at 0x000001FE19150E00> | callable: True | class object: False
<built-in function len> | callable: True | class object: False
<class 'int'> | callable: True | class object: True
42 | callable: False | class object: False


#### Solution

Every class in this example is callable.

But not every callable is a class.

That distinction matters when writing factory or registry code.

### Problem 7 — A Function That Accepts Only Class Objects

Suppose we want to write a factory.

The factory will accept a class object and create one new instance from it.

We will call the function `build_one`.

The first requirement is that the argument really is a class object.

In [35]:
def build_one(cls):
    if not isinstance(cls, type):
        raise TypeError("build_one expects a class object")

    return cls()

Now let's create a few classes that require no constructor arguments.

In [36]:
class North:
    pass

class South:
    pass

class East:
    pass

We can pass those class objects to our factory.

In [37]:
north = build_one(North)
south = build_one(South)
east = build_one(East)

Now verify each result.

In [38]:
print(type(north) is North)
print(type(south) is South)
print(type(east) is East)

True
True
True


Built-in classes can also work if calling them with no arguments is valid.

In [39]:
print(build_one(list))
print(build_one(dict))
print(build_one(set))

[]
{}
set()


Finally, let's verify that an instance is rejected as input.

In [40]:
try:
    build_one(North())
except TypeError as ex:
    print(type(ex).__name__ + ":", ex)

TypeError: build_one expects a class object


#### Solution

The key idea is that the function receives the class object itself, not the name of the class and not an already-created instance.

Because classes are callable, the factory can simply call `cls()` after validation.

### Problem 8 — Building Many Instances From One Class

Now let's extend the previous factory.

We want:

```python
build_many(cls, count)
```

to create a list containing `count` new objects created by `cls`.

Let's first write the core operation without validation.

In [41]:
def build_many(cls, count):
    return [cls() for _ in range(count)]

Try it with a simple class.

In [42]:
class Message:
    pass

messages = build_many(Message, 4)

messages

We should verify that every item has the correct class.

In [43]:
[type(item) is Message for item in messages]

[True, True, True, True]

And we should verify that they are separate objects rather than the same object repeated four times.

In [44]:
[id(item) for item in messages]

[2190853469392, 2190583401360, 2190854079184, 2190853769072]

Those identities should be different.

Now let's improve the function.

We should validate both arguments.

In [45]:
def build_many(cls, count):
    if not isinstance(cls, type):
        raise TypeError("cls must be a class object")

    if not isinstance(count, int):
        raise TypeError("count must be an integer")

    if count < 0:
        raise ValueError("count cannot be negative")

    return [cls() for _ in range(count)]

Let's test the valid case again.

In [46]:
messages = build_many(Message, 4)

print(len(messages))
print(all(isinstance(item, Message) for item in messages))
print(len({id(item) for item in messages}) == len(messages))

4
True
True


#### Solution

The three printed values are all evidence for different requirements:

- the requested number of objects was created
- every object is an instance of `Message`
- every object has a unique identity

### Problem 9 — Build a Class Registry Step by Step

A registry stores class objects so that we can look them up later.

We will build one using a dictionary.

The class name will be the key.

The class object itself will be the value.

In [47]:
registry = {}

Let's create three classes.

In [48]:
class CsvReader:
    pass

class JsonReader:
    pass

class TextReader:
    pass

A class gives us its name through `__name__`.

So we can store the first class like this.

In [49]:
registry[CsvReader.__name__] = CsvReader

registry

{'CsvReader': __main__.CsvReader}

Now add the other two.

In [50]:
registry[JsonReader.__name__] = JsonReader
registry[TextReader.__name__] = TextReader

registry

{'CsvReader': __main__.CsvReader,
 'JsonReader': __main__.JsonReader,
 'TextReader': __main__.TextReader}

Notice what we stored.

We stored `CsvReader`, not `CsvReader()`.

That difference is essential.

The registry should contain class objects so that we can call them later.

Let's look one up by name.

In [51]:
selected_class = registry["JsonReader"]

selected_class

__main__.JsonReader

And now call the retrieved class object.

In [52]:
reader = selected_class()

type(reader)

__main__.JsonReader

Now we can wrap registration in a function.

In [53]:
registry = {}

def register(cls):
    if not isinstance(cls, type):
        raise TypeError("Only class objects can be registered")

    registry[cls.__name__] = cls

Register all three classes.

In [54]:
register(CsvReader)
register(JsonReader)
register(TextReader)

registry

{'CsvReader': __main__.CsvReader,
 'JsonReader': __main__.JsonReader,
 'TextReader': __main__.TextReader}

And add a second function that creates an instance by registered name.

In [55]:
def create(name):
    cls = registry[name]
    return cls()

Let's try it.

In [56]:
obj = create("TextReader")

print(obj)
print(type(obj))
print(isinstance(obj, TextReader))

<class '__main__.TextReader'>
True


#### Solution

A registry like this has two useful levels:

```text
"text name"  -->  class object  -->  instance object
```

The dictionary connects the first two levels.

Calling the stored class creates the third.

### Problem 10 — Make the Registry Safer

The previous registry works, but it has two weaknesses.

First, registering a class with an existing name silently overwrites the previous entry.

Second, asking for an unknown class name produces a raw `KeyError`.

Let's improve both behaviors.

We will start with duplicate detection.

In [57]:
safe_registry = {}

def safe_register(cls):
    if not isinstance(cls, type):
        raise TypeError("Only class objects can be registered")

    name = cls.__name__

    if name in safe_registry:
        raise ValueError("A class named " + repr(name) + " is already registered")

    safe_registry[name] = cls

Let's test it.

In [58]:
class Parser:
    pass

safe_register(Parser)

try:
    safe_register(Parser)
except ValueError as ex:
    print(type(ex).__name__ + ":", ex)

ValueError: A class named 'Parser' is already registered


Now let's add a safe creation function.

When the name is unknown, we will show which names are available.

In [59]:
def safe_create(name):
    if name not in safe_registry:
        available = ", ".join(sorted(safe_registry))
        raise KeyError(
            "Unknown class "
            + repr(name)
            + ". Available: "
            + (available or "<none>")
        )

    return safe_registry[name]()

Test the valid case.

In [60]:
created = safe_create("Parser")

print(type(created))
print(isinstance(created, Parser))

<class '__main__.Parser'>
True


And now the invalid case.

In [61]:
try:
    safe_create("MissingParser")
except KeyError as ex:
    print(type(ex).__name__ + ":", ex)

KeyError: "Unknown class 'MissingParser'. Available: Parser"


#### Solution

The important design idea is that validation should happen close to the place where an invalid value enters the program.

That produces errors that are easier to understand than failures much later.

### Problem 11 — Dynamically Create a Class With `type`

So far we have used the `class` keyword.

But `type` can also create a new class object.

The three-argument form is:

```python
type(name, bases, namespace)
```

For this notebook we will use an empty tuple for `bases`, because inheritance is outside our current scope.

Let's create the equivalent of:

```python
class Generated:
    pass
```

In [62]:
Generated = type("Generated", (), {})

Now check what kind of object was returned.

In [63]:
type(Generated)

type

And check its name.

In [64]:
Generated.__name__

'Generated'

Because the returned object is a class, it should also be callable.

In [65]:
callable(Generated)

True

So we can create an instance normally.

In [66]:
g = Generated()

And the normal instance relationships still work.

In [67]:
print(type(g) is Generated)
print(g.__class__ is Generated)
print(isinstance(g, Generated))

True
True
True


#### Solution

A dynamically created class is still a normal class object for the purposes we are studying.

It has a name, it is callable, and its instances point back to it as their class.

### Problem 12 — Dynamically Add Class Attributes

The third argument to `type` is the namespace dictionary.

Entries in that dictionary become attributes on the new class.

Let's use that to create a class with some data.

In [68]:
Settings = type(
    "Settings",
    (),
    {
        "mode": "production",
        "retries": 3,
    }
)

We can access those attributes directly from the class.

In [69]:
Settings.mode

'production'

In [70]:
Settings.retries

3

And an instance can also access them.

In [71]:
settings = Settings()

print(settings.mode)
print(settings.retries)

production
3


Now let's inspect the relationship between the instance and the dynamically generated class.

In [72]:
print(type(settings) is Settings)
print(settings.__class__ is Settings)
print(isinstance(settings, Settings))
print(isinstance(Settings, type))

True
True
True
True


#### Solution

The dictionary supplied to `type` defines the initial namespace of the class.

For the concepts in this notebook, the generated class behaves just like one created with a normal `class` statement.

### Problem 13 — Dynamically Add a Method

Functions can also be placed in the namespace dictionary.

When accessed through an instance, they behave as methods.

Let's define a normal function first.

In [73]:
def describe(self):
    return "instance of " + self.__class__.__name__

Now place that function into a dynamically created class.

In [74]:
Describer = type(
    "Describer",
    (),
    {
        "describe": describe,
    }
)

Create an instance.

In [75]:
d = Describer()

And call the method through the instance.

In [76]:
d.describe()

'instance of Describer'

The method used `self.__class__.__name__`.

So it discovered the name of the class from the instance it received.

Let's inspect that in smaller steps.

In [77]:
d.__class__

__main__.Describer

In [78]:
d.__class__.__name__

'Describer'

#### Solution

The expression:

```python
self.__class__.__name__
```

moves through two objects:

1. from the instance to its class object
2. from the class object to its name string

### Problem 14 — Generate a Family of Unrelated Classes

Suppose a program receives class names as data.

We want to turn each name into a new class object.

Start with these names.

In [79]:
names = [
    "Bronze",
    "Silver",
    "Gold",
    "Platinum",
]

We can create one class from one name.

In [80]:
Bronze = type(names[0], (), {})

Bronze

__main__.Bronze

Now generalize that operation with a list comprehension.

In [81]:
classes = [
    type(name, (), {})
    for name in names
]

classes

[__main__.Bronze, __main__.Silver, __main__.Gold, __main__.Platinum]

Each item in `classes` should itself be a class object.

In [82]:
[isinstance(cls, type) for cls in classes]

[True, True, True, True]

And each class should preserve the name we supplied.

In [83]:
[cls.__name__ for cls in classes]

['Bronze', 'Silver', 'Gold', 'Platinum']

Now create one instance from every class.

In [84]:
objects = [
    cls()
    for cls in classes
]

objects

Let's compare each class with the corresponding instance.

In [85]:
for cls, obj in zip(classes, objects):
    print(
        cls.__name__,
        "| type(obj) is cls:",
        type(obj) is cls,
        "| isinstance:",
        isinstance(obj, cls)
    )

Bronze | type(obj) is cls: True | isinstance: True
Silver | type(obj) is cls: True | isinstance: True
Gold | type(obj) is cls: True | isinstance: True
Platinum | type(obj) is cls: True | isinstance: True


#### Solution

This example shows why classes being ordinary objects is useful.

They can be:

- created dynamically
- stored in a list
- iterated over
- called later to create instances

### Problem 15 — Build an Introspection Function in Small Steps

Let's write a function that reports useful information about any object.

Instead of writing the whole function immediately, we will build one piece at a time.

First, the runtime type.

In [86]:
def inspect_value(value):
    return {
        "type": type(value),
    }

In [87]:
inspect_value(100)

{'type': int}

Next, add the object's `__class__`.

In [88]:
def inspect_value(value):
    return {
        "type": type(value),
        "__class__": value.__class__,
    }

For the kinds of objects in this notebook, those should identify the same class object.

Let's record that too.

In [89]:
def inspect_value(value):
    return {
        "type": type(value),
        "__class__": value.__class__,
        "same_class_object": type(value) is value.__class__,
    }

Now add whether the value is callable and whether it is a class object.

In [90]:
def inspect_value(value):
    return {
        "type": type(value),
        "__class__": value.__class__,
        "same_class_object": type(value) is value.__class__,
        "callable": callable(value),
        "is_class_object": isinstance(value, type),
    }

Finally, we would like to include `__name__` when the object has one.

Not every object has that attribute, so `getattr` is convenient.

In [91]:
def inspect_value(value):
    return {
        "type": type(value),
        "__class__": value.__class__,
        "same_class_object": type(value) is value.__class__,
        "callable": callable(value),
        "is_class_object": isinstance(value, type),
        "__name__": getattr(value, "__name__", None),
    }

Let's test several fundamentally different kinds of values.

In [92]:
class Example:
    pass

def function_example():
    pass

test_values = [
    Example,
    Example(),
    type,
    int,
    10,
    "hello",
    function_example,
]

for value in test_values:
    print("=" * 60)
    print(repr(value))
    print(inspect_value(value))

<class '__main__.Example'>
{'type': <class 'type'>, '__class__': <class 'type'>, 'same_class_object': True, 'callable': True, 'is_class_object': True, '__name__': 'Example'}
{'type': <class '__main__.Example'>, '__class__': <class '__main__.Example'>, 'same_class_object': True, 'callable': False, 'is_class_object': False, '__name__': None}
<class 'type'>
{'type': <class 'type'>, '__class__': <class 'type'>, 'same_class_object': True, 'callable': True, 'is_class_object': True, '__name__': 'type'}
<class 'int'>
{'type': <class 'type'>, '__class__': <class 'type'>, 'same_class_object': True, 'callable': True, 'is_class_object': True, '__name__': 'int'}
10
{'type': <class 'int'>, '__class__': <class 'int'>, 'same_class_object': True, 'callable': False, 'is_class_object': False, '__name__': None}
'hello'
{'type': <class 'str'>, '__class__': <class 'str'>, 'same_class_object': True, 'callable': False, 'is_class_object': False, '__name__': None}
<function function_example at 0x000001FE1915176

#### Solution

The final function combines several small introspection operations.

The important part is not the dictionary itself.

The important part is understanding what each field means and which object level it describes.

### Problem 16 — Debug a Registry That Stores the Wrong Thing

Consider this registry code:

```python
registry["Task"] = Task()
```

Later the program tries to do:

```python
registry["Task"]()
```

Why does that design fail?

Let's reproduce the problem.

In [93]:
class Task:
    pass

bad_registry = {
    "Task": Task(),
}

Look at what is actually stored.

In [94]:
bad_registry["Task"]

Now ask whether that stored object is callable.

In [95]:
callable(bad_registry["Task"])

False

It is an instance, not the class object.

So calling it does not perform normal class instantiation.

The registry should store this instead:

In [96]:
good_registry = {
    "Task": Task,
}

Now the stored value is a class object.

In [97]:
isinstance(good_registry["Task"], type)

True

And class objects are callable.

In [98]:
callable(good_registry["Task"])

True

So this works.

In [99]:
new_task = good_registry["Task"]()

print(new_task)
print(type(new_task))
print(isinstance(new_task, Task))

<class '__main__.Task'>
True


#### Solution

The bug comes from confusing:

```python
Task
```

with:

```python
Task()
```

The first is the class object.

The second is an instance returned after calling that class.

### Problem 17 — A More Subtle Identity Puzzle

Let's finish with a prediction problem.

Do not run the next code immediately.

Read each expression and decide whether you expect `True` or `False`.

In [100]:
class Node:
    pass

a = Node()
b = Node()
c = a

Predict these:

```python
a is b
a is c
type(a) is Node
type(b) is Node
a.__class__ is b.__class__
isinstance(a, Node)
isinstance(Node, type)
isinstance(Node, Node)
type(Node) is type
type(type) is type
```

Now run them one at a time.

In [101]:
a is b

False

In [102]:
a is c

True

In [103]:
type(a) is Node

True

In [104]:
type(b) is Node

True

In [105]:
a.__class__ is b.__class__

True

In [106]:
isinstance(a, Node)

True

In [107]:
isinstance(Node, type)

True

In [108]:
isinstance(Node, Node)

False

In [109]:
type(Node) is type

True

In [110]:
type(type) is type

True

Let's put the results together so we can see the pattern clearly.

In [111]:
checks = [
    ("a is b", a is b),
    ("a is c", a is c),
    ("type(a) is Node", type(a) is Node),
    ("type(b) is Node", type(b) is Node),
    ("a.__class__ is b.__class__", a.__class__ is b.__class__),
    ("isinstance(a, Node)", isinstance(a, Node)),
    ("isinstance(Node, type)", isinstance(Node, type)),
    ("isinstance(Node, Node)", isinstance(Node, Node)),
    ("type(Node) is type", type(Node) is type),
    ("type(type) is type", type(type) is type),
]

for expression, result in checks:
    print(expression.ljust(34), "->", result)

a is b                             -> False
a is c                             -> True
type(a) is Node                    -> True
type(b) is Node                    -> True
a.__class__ is b.__class__         -> True
isinstance(a, Node)                -> True
isinstance(Node, type)             -> True
isinstance(Node, Node)             -> False
type(Node) is type                 -> True
type(type) is type                 -> True


#### Solution

The most important distinction is again the level we are talking about.

`a` and `b` are instances of `Node`.

`Node` itself is a class object.

So:

```python
isinstance(a, Node)
```

is true, while:

```python
isinstance(Node, Node)
```

is false.

But because `Node` is a class object:

```python
isinstance(Node, type)
```

is true.

### Problem 18 — Final Tutorial Challenge: Class Catalog

We will now combine the ideas from the entire notebook.

We want to build a small **class catalog**.

The catalog should:

1. store class objects by name
2. reject non-class values
3. reject duplicate names
4. list available class names
5. create an instance by name
6. report information about every stored class

We will build it one function at a time.

First, create the dictionary.

In [112]:
catalog = {}

Now write `add_class`.

We already know the first validation:

```python
isinstance(cls, type)
```

In [113]:
def add_class(cls):
    if not isinstance(cls, type):
        raise TypeError("Only class objects can be added")

    name = cls.__name__

    if name in catalog:
        raise ValueError("Duplicate class name: " + name)

    catalog[name] = cls

Let's create five classes for the catalog.

In [114]:
class Upload:
    pass

class Download:
    pass

class Compress:
    pass

class Validate:
    pass

class Archive:
    pass

Add them.

In [115]:
for cls in [Upload, Download, Compress, Validate, Archive]:
    add_class(cls)

catalog

{'Upload': __main__.Upload,
 'Download': __main__.Download,
 'Compress': __main__.Compress,
 'Validate': __main__.Validate,
 'Archive': __main__.Archive}

Now make a function that lists the names.

Sorting gives us predictable output.

In [116]:
def available_classes():
    return sorted(catalog)

available_classes()

['Archive', 'Compress', 'Download', 'Upload', 'Validate']

Next, create an instance by name.

The dictionary gives us the class object.

Then we call that class object.

In [117]:
def create_from_catalog(name):
    if name not in catalog:
        raise KeyError("Unknown class: " + repr(name))

    cls = catalog[name]

    return cls()

Let's create one object.

In [118]:
upload = create_from_catalog("Upload")

print(upload)
print(type(upload))
print(isinstance(upload, Upload))

<class '__main__.Upload'>
True


Now create one instance of every registered class.

In [119]:
created_objects = {}

for name in available_classes():
    created_objects[name] = create_from_catalog(name)

created_objects

{'Archive': <__main__.Archive at 0x1fe191b0980>,
 'Compress': <__main__.Compress at 0x1fe191b0c20>,
 'Download': <__main__.Download at 0x1fe191b0d70>,
 'Upload': <__main__.Upload at 0x1fe19144910>,
 'Validate': <__main__.Validate at 0x1fe191b0ec0>}

Let's verify every class-instance pair.

In [120]:
for name in available_classes():
    cls = catalog[name]
    obj = created_objects[name]

    print(
        name,
        "| class object:",
        isinstance(cls, type),
        "| object has exact class:",
        type(obj) is cls,
        "| isinstance:",
        isinstance(obj, cls)
    )

Archive | class object: True | object has exact class: True | isinstance: True
Compress | class object: True | object has exact class: True | isinstance: True
Download | class object: True | object has exact class: True | isinstance: True
Upload | class object: True | object has exact class: True | isinstance: True
Validate | class object: True | object has exact class: True | isinstance: True


Finally, write a small report function.

For each stored class, show:

- its name
- its type
- whether it is callable
- whether it is an instance of `type`

In [121]:
def catalog_report():
    for name in available_classes():
        cls = catalog[name]

        print(
            "name =", name,
            "| type =", type(cls).__name__,
            "| callable =", callable(cls),
            "| class object =", isinstance(cls, type)
        )

catalog_report()

name = Archive | type = type | callable = True | class object = True
name = Compress | type = type | callable = True | class object = True
name = Download | type = type | callable = True | class object = True
name = Upload | type = type | callable = True | class object = True
name = Validate | type = type | callable = True | class object = True


Now test the error cases too.

First, a duplicate class.

In [122]:
try:
    add_class(Upload)
except ValueError as ex:
    print(type(ex).__name__ + ":", ex)

ValueError: Duplicate class name: Upload


Next, an instance instead of a class object.

In [123]:
try:
    add_class(Upload())
except TypeError as ex:
    print(type(ex).__name__ + ":", ex)

TypeError: Only class objects can be added


And finally, an unknown name.

In [124]:
try:
    create_from_catalog("Missing")
except KeyError as ex:
    print(type(ex).__name__ + ":", ex)

KeyError: "Unknown class: 'Missing'"


#### Final Solution

The catalog works because classes are ordinary Python objects.

We can store them in a dictionary, inspect them, retrieve them, and call them.

The flow is:

```text
name string
    |
    v
class object
    |
    | call it
    v
instance object
```

That small pattern appears in many larger Python systems.

### Extra Guided Practice

The remaining exercises are shorter, but they should still be solved in the same way:

1. predict the behavior
2. write the smallest possible code
3. inspect the result
4. explain which object is a class and which object is an instance

#### Practice 1

Write a function:

```python
class_name(value)
```

that returns the name of the runtime class of any object.

For example:

```python
class_name(10)       # "int"
class_name("hello")  # "str"
```

A useful starting point is:

```python
type(value)
```

That gives the class object.

Then the class object has `__name__`.

In [125]:
def class_name(value):
    return type(value).__name__

print(class_name(10))
print(class_name("hello"))
print(class_name([]))

int
str
list


#### Practice 2

Write:

```python
same_class(a, b)
```

that returns `True` only when the two values have the exact same runtime class object.

We can get each runtime class with `type`.

Then compare the class objects with `is`.

In [126]:
def same_class(a, b):
    return type(a) is type(b)

print(same_class(10, 20))
print(same_class(10, 20.0))
print(same_class([], []))
print(same_class([], {}))

True
False
True
False


#### Practice 3

Given a mixed list of class objects and instances, keep only the class objects.

In [127]:
class A:
    pass

class B:
    pass

mixed = [
    A,
    A(),
    B,
    B(),
    int,
    10,
    str,
    "x",
]

We already have the exact test we need:

```python
isinstance(value, type)
```

In [128]:
only_classes = [
    value
    for value in mixed
    if isinstance(value, type)
]

only_classes

[__main__.A, __main__.B, int, str]

#### Practice 4

Now keep only the values that are callable.

Remember that this is not the same question as keeping only classes.

In [129]:
def example_function():
    pass

mixed_callables = [
    A,
    A(),
    int,
    10,
    example_function,
    len,
]

only_callables = [
    value
    for value in mixed_callables
    if callable(value)
]

only_callables

[__main__.A,
 int,
 <function __main__.example_function()>,
 <function len(obj, /)>]

Notice that the result contains both classes and functions.

That is exactly why `callable` and `isinstance(value, type)` should not be treated as interchangeable tests.

### Summary

A class is itself an object.

For the normal classes used in this notebook:

```python
type(SomeClass) is type
```

and:

```python
isinstance(SomeClass, type)
```

are true.

Calling a class creates an instance:

```python
obj = SomeClass()
```

Then:

```python
type(obj) is SomeClass
obj.__class__ is SomeClass
isinstance(obj, SomeClass)
```

are all useful ways of inspecting the relationship.

`__name__` gives us the name of a class as text.

That is useful for display, logging, and dictionary keys.

But a name string is not a substitute for the actual class object when checking types.

Classes are callable, but functions are callable too.

So:

```python
callable(value)
```

and:

```python
isinstance(value, type)
```

answer different questions.

Finally, `type` itself can create class objects:

```python
type(name, bases, namespace)
```

That lets us generate classes dynamically while preserving the same relationships between class objects and their instances.

There is much more to say once inheritance and metaprogramming are introduced.

At that point, ideas such as `isinstance`, method resolution, and the behavior of `type` become more subtle.

For now, the goal is to be completely comfortable with the two levels we used throughout this notebook:

```text
class object  <---->  instance object
```